In [ ]:
import pandas as pd

### 1.1 blinkit_products.csv

Bộ dữ liệu `blinkit_products.csv` là bảng danh mục sản phẩm. Bảng này đóng vai trò làm cơ sở tham chiếu cốt lõi để liên kết với các bảng dữ liệu giao dịch nghiệp vụ khác trong hệ thống của Blinkit.

Bộ dữ liệu gồm 10 biến như sau:
- Biến định danh và phân loại (Định tính):<br>
`product_id`: Mã định danh sản phẩm (Khóa chính).<br>
`product_name`: Tên nhóm sản phẩm.<br>
`category`: Ngành hàng.<br>
`brand`: Thương hiệu cung cấp.

- Biến tài chính (Định lượng):<br>
`price`: Giá vốn.<br>
`mrp`: Giá bán lẻ tối đa.<br>
`margin_percentage`: Tỷ lệ biên lợi nhuận, tính bằng công thức (`mrp`-`price`)/`mrp`.

- Biến quản lý lưu kho (Định lượng):<br>
`shelf_life_days`: Thời hạn lưu kho tối đa.<br>
`min_stock_level`: Định mức tồn kho tối thiểu.<br>
`max_stock_level`: Định mức tồn kho tối đa.

In [ ]:
# Đọc dữ liệu
df_products = pd.read_csv('blinkit_products.csv')

# Kiểm tra quy mô
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_products.shape)

# Xem 5 dòng đầu tiên để đối chiếu
df_products.head()

Quy mô bộ dữ liệu (Dòng, Cột): (268, 10)


,product_id,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level
0,153019,Onions,Fruits & Vegetables,Aurora LLC,947.95,1263.93,25.0,3,13,88
1,11422,Potatoes,Fruits & Vegetables,Ramaswamy-Tata,127.16,169.55,25.0,3,20,65
2,669378,Potatoes,Fruits & Vegetables,Chadha and Sons,212.14,282.85,25.0,3,23,70
3,848226,Tomatoes,Fruits & Vegetables,Barad and Sons,209.59,279.45,25.0,3,10,51
4,890623,Onions,Fruits & Vegetables,"Sangha, Nagar and Varty",354.52,472.69,25.0,3,27,55


Bộ dữ liệu bao gồm 268 bản ghi và 10 biến.

In [ ]:
# Xây dựng bảng tổng hợp chất lượng dữ liệu song song
df_products_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_products.dtypes.astype(str),
    'Số lượng giá trị Null': df_products.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_products.nunique()
})

# Hiển thị bảng tổng hợp
df_products_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
product_id,int64,0,268
product_name,object,0,51
category,object,0,11
brand,object,0,267
price,float64,0,267
mrp,float64,0,268
margin_percentage,float64,0,6
shelf_life_days,int64,0,5
min_stock_level,int64,0,21
max_stock_level,int64,0,51


1. Độ đầy đủ: <br>
Toàn bộ 10 biến đều hoàn thiện, không ghi nhận giá trị khuyết thiếu (Null = 0).

2. Tính duy nhất: <br>
Biến `product_id` chứa đúng 268 giá trị phân biệt tương ứng với 268 bản ghi, đủ điều kiện làm khóa chính (Primary Key).

3. Đặc điểm phân phối cơ bản: <br>
Hệ thống bao gồm 11 ngành hàng (`category`) và 51 nhóm sản phẩm (`product_name`).<br>
Có 267 thương hiệu (`brand`) cung cấp, trong đó thương hiệu "Jha Group" xuất hiện 2 lần.<br>
Có mức giá (`price`) 826,21 xuất hiện 2 lần do thương hiệu "Raghavan, Magar and Shanker" và "Kunda, Dave and Sahni" cung cấp.<br>
Đặc điểm này cho thấy sự đa dạng của danh mục hàng hóa phụ thuộc vào số lượng thương hiệu cung ứng thay vì số lượng nhóm sản phẩm.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")
num_cols = ['price', 'mrp', 'margin_percentage', 'shelf_life_days', 'min_stock_level', 'max_stock_level']

# 1. Kiểm tra giá trị <= 0 ở các biến định lượng bắt buộc dương
for col in num_cols:
    invalid_count = (df_products[col] <= 0).sum()
    print(f"Biến `{col}` có giá trị <= 0: {invalid_count} bản ghi.")

# 2. Kiểm tra logic tài chính (Price phải nhỏ hơn hoặc bằng MRP)
price_gt_mrp = (df_products['price'] > df_products['mrp']).sum()
pct_price_gt_mrp = (price_gt_mrp / len(df_products)) * 100
print(f"Giá bán thực tế lớn hơn MRP đề xuất: {price_gt_mrp} bản ghi ({pct_price_gt_mrp:.2f}%).")

# 3. Kiểm tra logic tồn kho (Min Stock phải nhỏ hơn hoặc bằng Max Stock)
min_gt_max = (df_products['min_stock_level'] > df_products['max_stock_level']).sum()
print(f"Tồn kho tối thiểu lớn hơn tồn kho tối đa: {min_gt_max} bản ghi.")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `price` có giá trị <= 0: 0 bản ghi.
Biến `mrp` có giá trị <= 0: 0 bản ghi.
Biến `margin_percentage` có giá trị <= 0: 0 bản ghi.
Biến `shelf_life_days` có giá trị <= 0: 0 bản ghi.
Biến `min_stock_level` có giá trị <= 0: 0 bản ghi.
Biến `max_stock_level` có giá trị <= 0: 0 bản ghi.
Giá bán thực tế lớn hơn MRP đề xuất: 0 bản ghi (0.00%).
Tồn kho tối thiểu lớn hơn tồn kho tối đa: 0 bản ghi.


Nhận xét về giá trị bất thường: 
1. Tính toàn vẹn của giá trị số: <br>
Không có bất kỳ biến số định lượng nào (`price`, `mrp`, `margin_percentage`, `shelf_life_days`, `min_stock_level`, `max_stock_level`) chứa giá trị âm hoặc bằng 0 vô lý.

2. Logic ràng buộc tài chính: <br>
Ghi nhận 0 trường hợp giá bán thực tế (`price`) vượt quá mức giá bán lẻ tối đa đề xuất (`mrp`). Toàn bộ hệ thống giá phân phối tuân thủ chặt chẽ ràng buộc kinh doanh.

3. Logic ràng buộc tham số kho: <br>
Toàn bộ bản ghi đều tuân thủ nguyên tắc cấu trúc kho, không xuất hiện lỗi định mức tồn kho tối thiểu (`min_stock_level`) vượt ngưỡng định mức tối đa (`max_stock_level`).

In [ ]:
# Xuất bảng thống kê mô tả sau khi xác nhận dữ liệu không có bất thường logic (làm tròn 2 chữ số thập phân)
df_products[num_cols].describe().round(2)

,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level
count,268.00,268.00,268.00,268.00,268.00,268.00
mean,488.36,680.43,27.78,231.76,20.39,74.75
std,298.49,419.77,7.46,151.21,5.96,14.59
min,12.32,17.60,15.00,3.00,10.00,50.00
25%,226.72,325.15,20.00,90.00,15.00,63.75
50%,442.18,616.97,30.00,365.00,21.00,73.00
75%,779.44,1056.62,35.00,365.00,25.25,88.00
max,995.98,1633.32,40.00,365.00,30.00,100.00


Nhận xét về các chỉ số thống kê toàn danh mục:
1. Nhóm biến tài chính: <br>
Các chỉ số trung bình (mean) và trung vị (50%) của `price` và `mrp` phản ánh mức giá phân phối chung của các mặt hàng. Khoảng cách giữa các phân vị (25%, 50%, 75%) cho thấy mức độ phân hóa giá giữa các nhóm mặt hàng bình dân và cao cấp.<br>
Biến `margin_percentage` có độ lệch chuẩn thấp, cho thấy tỷ lệ biên lợi nhuận được áp đặt khá đồng đều và ổn định trên toàn bộ danh mục sản phẩm, cụ thể:
2. Nhóm biến lưu kho: <br>
Biến `shelf_life_days` có giá trị trung vị giúp nhận diện vòng đời lưu kho phổ biến của hàng hóa (đặc thù của ngành giao hàng nhanh FMCG thường có chu kỳ ngắn).<br>
Các ngưỡng `min_stock_level` và `max_stock_level` thể hiện quy mô sức chứa thiết lập tại kho. Sự chênh lệch giữa giá trị lớn nhất (max) và nhỏ nhất (min) của hai biến này phản ánh mức độ ưu tiên lưu kho giữa các mặt hàng nhu yếu phẩm (bán chạy) và các mặt hàng phụ trợ.

In [ ]:
summary_df = df_products.groupby('category').agg({
    'margin_percentage': 'mean',
    'shelf_life_days': 'max'
}).reset_index()

summary_df['margin_percentage'] = summary_df['margin_percentage'].round(0).astype(int)

summary_df

,category,margin_percentage,shelf_life_days
0,Baby Care,30,365
1,Cold Drinks & Juices,30,180
2,Dairy & Breakfast,20,7
3,Fruits & Vegetables,25,3
4,Grocery & Staples,15,365
5,Household Care,25,365
6,Instant & Frozen Food,40,180
7,Personal Care,35,365
8,Pet Care,35,365
9,Pharmacy,20,365


Nhận xét:
- Lợi nhuận biên và thời gian lưu kho tối đa là cố định với mỗi danh mục sản phẩm.

### 1.2 blinkit_customers.csv

Bộ dữ liệu `blinkit_customers.csv` là bảng danh mục khách hàng (Dimension Table). Bảng này đóng vai trò lưu trữ thông tin định danh, các phương thức liên lạc, vị trí địa lý bưu chính và các chỉ số hành vi tiêu dùng cơ bản của toàn bộ tệp khách hàng đăng ký trên hệ thống của Blinkit.

Bộ dữ liệu gồm 11 biến như sau:
- Biến định danh và liên lạc (Định tính):<br>
`customer_id`: Mã định danh duy nhất của từng khách hàng (Khóa chính).<br>
`customer_name`: Họ và tên đầy đủ của khách hàng.<br>
`email`: Địa chỉ thư điện tử cá nhân.<br>
`phone`: Số điện thoại liên lạc đăng ký trên hệ thống.

- Biến vị trí địa lý (Định tính):<br>
`address`: Địa chỉ cư trú chi tiết.<br>
`area`: Khu vực hoặc thành phố hành chính.<br>
`pincode`: Mã bưu chính tương ứng với khu vực địa lý.

- Biến phân loại và thời gian (Định tính):<br>
`registration_date`: Ngày khách hàng đăng ký tài khoản trên hệ thống.<br>
`customer_segment`: Phân khúc phân loại nhóm khách hàng có sẵn.<br>

- Biến hành vi tiêu dùng (Định lượng):<br>
`total_orders`: Tổng số lượng đơn hàng khách hàng đã thực hiện trong lịch sử.<br>
`avg_order_value`: Giá trị trung bình của một đơn hàng mà khách hàng chi trả.

In [ ]:
# Đọc dữ liệu khách hàng
df_customers = pd.read_csv('blinkit_customers.csv')

# Kiểm tra quy mô
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_customers.shape)

# Xem 5 dòng đầu tiên để đối chiếu
df_customers.head()

Quy mô bộ dữ liệu (Dòng, Cột): (2500, 11)


,customer_id,customer_name,email,phone,address,area,pincode,registration_date,customer_segment,total_orders,avg_order_value
0,97475543,Niharika Nagi,ektataneja@example.org,912987579691,"23, Nayar Path, Bihar Sharif-154625",Udupi,321865,2023-05-13,Premium,13,451.92
1,22077605,Megha Sachar,vedant45@example.com,915123179717,"51/302, Buch Chowk\nSrinagar-570271",Aligarh,149394,2024-06-18,Inactive,4,825.48
2,47822591,Hema Bahri,samiazaan@example.com,910034076149,"941\nAnne Street, Darbhanga 186125",Begusarai,621411,2024-09-25,Regular,17,1969.81
3,79726146,Zaitra Vig,ishanvi87@example.org,916264232390,"43/94, Ghosh, Alappuzha 635655",Kozhikode,826054,2023-10-04,New,4,220.09
4,57102800,Januja Verma,atideshpande@example.org,917293526596,"06\nOm, Ambarnath 477463",Ichalkaranji,730539,2024-03-22,Inactive,14,578.14


Bộ dữ liệu bao gồm 2500 bản ghi và 11 biến.

In [ ]:
# Xây dựng bảng tổng hợp chất lượng dữ liệu song song cho bảng khách hàng
df_customers_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_customers.dtypes.astype(str),
    'Số lượng giá trị Null': df_customers.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_customers.nunique()
})

# Hiển thị bảng tổng hợp
df_customers_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
customer_id,int64,0,2500
customer_name,object,0,2491
email,object,0,2496
phone,int64,0,2500
address,object,0,2500
area,object,0,316
pincode,int64,0,2494
registration_date,object,0,589
customer_segment,object,0,4
total_orders,int64,0,20


Nhận xét: 
1. Độ đầy đủ: <br>
Toàn bộ 11 biến đều hoàn thiện thông tin, không ghi nhận bất kỳ giá trị khuyết thiếu nào trên toàn bảng dữ liệu (Null = 0).

2. Tính duy nhất: <br>
Biến `customer_id` chứa đúng 2500 giá trị phân biệt, hoàn toàn trùng khớp với tổng số dòng dữ liệu, đủ điều kiện cấu trúc để thiết lập làm khóa chính (Primary Key). Ngoài ra, trường `phone` và trường `address` cũng đạt độ phân biệt tuyệt đối (2500 giá trị duy nhất), đảm bảo không có hiện tượng trùng lặp thông tin liên lạc và vị trí giao hàng cá nhân.

3. Đặc điểm phân phối cơ bản: <br>
Tệp khách hàng của Blinkit được phân bổ trải rộng trên 316 khu vực hành chính (`area`) khác nhau.<br>
Biến `customer_segment` chứa 4 giá trị phân biệt (Inactive, New, Regular, Premium).<br>
Biến `total_orders` chỉ nhận 20 giá trị phân biệt, phản ánh số lượng đơn hàng tích lũy của mỗi cá nhân dao động trong một tập hợp hữu hạn các khoảng số nguyên cố định.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra giá trị âm ở các biến định lượng hành vi
print(f"Tổng số bản ghi có `total_orders` < 0: {(df_customers['total_orders'] < 0).sum()}")
print(f"Tổng số bản ghi có `avg_order_value` < 0: {(df_customers['avg_order_value'] < 0).sum()}")

# 2. Kiểm tra logic ràng buộc hành vi tiêu dùng chéo
# Nếu đơn hàng > 0 thì giá trị trung bình phải > 0 và ngược lại
order_error = ((df_customers['total_orders'] > 0) & (df_customers['avg_order_value'] <= 0)).sum()
value_error = ((df_customers['avg_order_value'] > 0) & (df_customers['total_orders'] <= 0)).sum()
print(f"Lỗi logic: Có đơn hàng nhưng giá trị bằng 0: {order_error} bản ghi.")
print(f"Lỗi logic: Có giá trị đơn hàng nhưng số lượng bằng 0: {value_error} bản ghi.")

# 3. Kiểm tra tính hợp lệ sơ bộ của chuỗi liên lạc (Email phải chứa ký tự '@')
email_invalid = (~df_customers['email'].str.contains('@', na=False)).sum()
print(f"Số lượng email sai định dạng (thiếu '@'): {email_invalid} bản ghi.")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Tổng số bản ghi có `total_orders` < 0: 0
Tổng số bản ghi có `avg_order_value` < 0: 0
Lỗi logic: Có đơn hàng nhưng giá trị bằng 0: 0 bản ghi.
Lỗi logic: Có giá trị đơn hàng nhưng số lượng bằng 0: 0 bản ghi.
Số lượng email sai định dạng (thiếu '@'): 0 bản ghi.


Nhận xét về giá trị bất thường: 
1. Tính toàn vẹn của giá trị số: <br>
Không xuất hiện giá trị âm ở cả hai biến định lượng hành vi tiêu dùng (`total_orders` và `avg_order_value`).

2. Logic ràng buộc hành vi chéo: <br>
Ghi nhận 0 trường hợp lỗi logic chéo. Toàn bộ khách hàng có số lượng đơn hàng lớn hơn 0 đều đi kèm với giá trị đơn hàng trung bình dương tương ứng, và ngược lại không có tài khoản nào phát sinh số tiền chi trả khi chưa từng đặt hàng.

3. Logic định dạng thông tin liên lạc: <br>
Toàn bộ 2500 chuỗi ký tự tại trường `email` đều chứa ký tự định dạng cấu trúc bắt buộc (`@`), sơ bộ đảm bảo không có lỗi nhập liệu thô ở thông tin liên lạc.

In [ ]:
# Lọc danh sách các trường định lượng hành vi (bỏ qua các trường mã số ID, Phone, Pincode)
cust_num_cols = ['total_orders', 'avg_order_value']

# Xuất bảng thống kê mô tả sau khi xác nhận dữ liệu sạch logic
df_customers[cust_num_cols].describe().round(2)

,total_orders,avg_order_value
count,2500.00,2500.00
mean,10.49,1102.38
std,5.81,523.04
min,1.00,200.43
25%,6.00,631.82
50%,10.00,1118.65
75%,16.00,1565.40
max,20.00,1999.83


Nhận xét về các chỉ số thống kê toàn tệp khách hàng:
1. Biến tổng số đơn hàng (`total_orders`): <br>
Các chỉ số phân vị và giá trị trung bình phác họa tần suất mua sắm tích lũy của tệp khách hàng. Khoảng cách giữa giá trị nhỏ nhất (min) và lớn nhất (max) phản ánh sự phân hóa về lòng trung thành hoặc thời gian gắn kết của các nhóm người dùng trên ứng dụng.<br>

2. Biến giá trị đơn hàng trung bình (`avg_order_value`): <br>
Giá trị trung bình và trung vị (50%) cung cấp cái nhìn về quy mô chi tiêu phổ biến của một giỏ hàng. Sự chênh lệch giữa các ngưỡng phân vị thấp (25%) và cao (75%) thể hiện sức mua đa dạng của tệp khách hàng, hỗ trợ nhận diện các nhóm khách hàng có giá trị cao đối với doanh thu hệ thống.

### 1.3 Category_Icons.csv

Bộ dữ liệu `Category_Icons.csv` là bảng danh mục biểu tượng ngành hàng (Dimension Table). Bảng này đóng vai trò lưu trữ đường dẫn tệp hoặc tên gọi của các biểu tượng thiết kế đồ họa đại diện cho từng ngành hàng, phục vụ việc đồng bộ hiển thị giao diện trực quan trên ứng dụng của Blinkit.

Bộ dữ liệu gồm 2 biến như sau:
- Biến định danh và phân loại (Định tính):<br>
`category`: Tên ngành hàng phân biệt (Khóa chính).<br>
`Img`: Tên tệp hoặc liên kết dẫn đến hình ảnh biểu tượng tương ứng của ngành hàng.

In [ ]:
# Đọc dữ liệu biểu tượng ngành hàng
df_cat_icons = pd.read_csv('Category_Icons.csv')

# Kiểm tra quy mô
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_cat_icons.shape)

# Xem toàn bộ dữ liệu (do quy mô bảng rất nhỏ)
df_cat_icons

Quy mô bộ dữ liệu (Dòng, Cột): (11, 2)


,category,Img
0,Grocery & Staples,https://i.postimg.cc/jSc7fmbs/grocery.png
1,Household Care,https://i.postimg.cc/1tnnLzNt/cleaning-tools.png
2,Personal Care,https://i.postimg.cc/hvmm7GSw/personal-care.png
3,Baby Care,https://i.postimg.cc/k55tvsJW/feeding.png
4,Pet Care,https://i.postimg.cc/FRMdmQjx/pet-food.png
5,Pharmacy,https://i.postimg.cc/W3x7FCP9/pharmacy.png
6,Fruits & Vegetables,https://i.postimg.cc/sgsWvyb6/fruits.png
7,Dairy & Breakfast,https://i.postimg.cc/X7wFjX1B/food.png
8,Snacks & Munchies,https://i.postimg.cc/CKP7Ydrp/snacks.png
9,Cold Drinks & Juices,https://i.postimg.cc/kXrKPWsd/softdrink.png


Bộ dữ liệu bao gồm 11 bản ghi và 2 biến.

In [ ]:
# Xây dựng bảng tổng hợp chất lượng dữ liệu song song cho bảng biểu tượng ngành hàng
df_cat_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_cat_icons.dtypes.astype(str),
    'Số lượng giá trị Null': df_cat_icons.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_cat_icons.nunique()
})

# Hiển thị bảng tổng hợp
df_cat_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
category,object,0,11
Img,object,0,11


Nhận xét: 
1. Độ đầy đủ: <br>
Toàn bộ dữ liệu hoàn thiện, không ghi nhận bất kỳ giá trị khuyết thiếu nào trên các trường thông tin (Null = 0).

2. Tính duy nhất: <br>
Biến `category` chứa đúng 11 giá trị phân biệt tương ứng với 11 dòng dữ liệu, đáp ứng đầy đủ điều kiện cấu trúc để làm khóa chính (Primary Key).


In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra chuỗi chỉ chứa toàn khoảng trắng (lỗi nhập liệu dấu cách ẩn)
blank_cat = (df_cat_icons['category'].str.strip() == "").sum()
blank_img = (df_cat_icons['Img'].str.strip() == "").sum()

# 2. Kiểm tra chuỗi bị dính khoảng trắng thừa ở đầu hoặc cuối chuỗi (gây lỗi khi nối bảng hoặc gọi link hình ảnh)
has_spaces_cat = (df_cat_icons['category'].str.startswith(' ') | df_cat_icons['category'].str.endswith(' ')).sum()
has_spaces_img = (df_cat_icons['Img'].str.startswith(' ') | df_cat_icons['Img'].str.endswith(' ')).sum()

print(f"Số lượng dòng có trường `category` chỉ chứa khoảng trắng: {blank_cat}")
print(f"Số lượng dòng có trường `Img` chỉ chứa khoảng trắng: {blank_img}")
print(f"Số lượng dòng có trường `category` bị dính khoảng trắng thừa ở đầu/cuối: {has_spaces_cat}")
print(f"Số lượng dòng có trường `Img` bị dính khoảng trắng thừa ở đầu/cuối: {has_spaces_img}")

# 3. Kiểm tra tính hợp lệ của phần mở rộng tệp hình ảnh tại trường `Img`
valid_extensions = (df_cat_icons['Img'].str.endswith(('.png', '.jpg', '.jpeg', '.svg', '.gif')))
invalid_extension_count = (~valid_extensions).sum()
print(f"Số lượng biểu tượng tại `Img` sai định dạng đuôi tệp hình ảnh: {invalid_extension_count}")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Số lượng dòng có trường `category` chỉ chứa khoảng trắng: 0
Số lượng dòng có trường `Img` chỉ chứa khoảng trắng: 0
Số lượng dòng có trường `category` bị dính khoảng trắng thừa ở đầu/cuối: 0
Số lượng dòng có trường `Img` bị dính khoảng trắng thừa ở đầu/cuối: 0
Số lượng biểu tượng tại `Img` sai định dạng đuôi tệp hình ảnh: 0


Nhận xét về giá trị bất thường: 
1. Tính toàn vẹn của chuỗi ký tự: <br>
Không phát hiện bản ghi nào chỉ chứa toàn ký tự khoảng trắng vô nghĩa. Đồng thời, toàn bộ chuỗi tại các trường định tính đều được chuẩn hóa sạch sẽ, không bị dính các ký tự khoảng trắng thừa ở đầu hoặc cuối chuỗi, đảm bảo tính khớp nối an toàn khi thực hiện nối bảng.

2. Logic định dạng tài nguyên đồ họa: <br>
Toàn bộ các bản ghi tại trường `Img` đều được cấu trúc chính xác, kết thúc bằng các định dạng tệp mở rộng tiêu chuẩn dành cho giao diện web/ứng dụng, sơ bộ loại trừ khả năng lỗi đường dẫn tài nguyên thô.

### 1.4 Rating_Icon.csv

Bộ dữ liệu `Rating_Icon.csv` là bảng danh mục biểu tượng đánh giá (Dimension Table). Bảng này đóng vai trò lưu trữ đường dẫn hình ảnh biểu cảm trực quan (Emoji) và ký tự hiển thị số sao (Star) tương ứng với từng mức điểm đánh giá của khách hàng trên hệ thống của Blinkit.

Bộ dữ liệu gồm 3 biến như sau:
- Biến định danh và phân loại (Định tính):<br>
`Rating`: Mức điểm đánh giá bằng số từ 1 đến 5 (Khóa chính).<br>
`Emoji`: Đường dẫn liên kết đến tệp hình ảnh biểu cảm tương ứng với mức điểm.<br>
`Star`: Chuỗi ký tự hiển thị số lượng ngôi sao đại diện cho mức điểm đánh giá.

In [ ]:
# Đọc dữ liệu biểu tượng đánh giá
df_rating_icons = pd.read_csv('Rating_Icon.csv', encoding='utf-8')

# Kiểm tra quy mô
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_rating_icons.shape)

# Xem toàn bộ dữ liệu (do quy mô bảng rất nhỏ)
df_rating_icons

Quy mô bộ dữ liệu (Dòng, Cột): (5, 3)


,Rating,Emoji,Star
0,1,https://i.postimg.cc/15zP0XBY/angry.png,⭐
1,2,https://i.postimg.cc/zGbrMQ9C/sad.png,⭐⭐
2,3,https://i.postimg.cc/d02zccny/confused.png,⭐⭐⭐
3,4,https://i.postimg.cc/X7XSGhNR/smile.png,⭐⭐⭐⭐
4,5,https://i.postimg.cc/XYyRxCCx/star.png,⭐⭐⭐⭐⭐


Bộ dữ liệu bao gồm 5 bản ghi và 3 biến.

In [ ]:
# Xây dựng bảng tổng hợp chất lượng dữ liệu kỹ thuật
df_rating_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_rating_icons.dtypes.astype(str),
    'Số lượng giá trị Null': df_rating_icons.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_rating_icons.nunique()
})

# Hiển thị bảng tổng hợp
df_rating_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
Rating,int64,0,5
Emoji,object,0,5
Star,object,0,5


Nhận xét: 
1. Độ đầy đủ: <br>
Toàn bộ dữ liệu hoàn thiện, không ghi nhận bất kỳ giá trị khuyết thiếu nào trên toàn bảng dữ liệu (Null = 0).

2. Tính duy nhất: <br>
Biến `Rating` chứa đúng 5 giá trị phân biệt tương ứng với 5 dòng dữ liệu, đáp ứng đầy đủ điều kiện cấu trúc để thiết lập làm khóa chính (Primary Key).

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra giá trị <= 0 hoặc vượt quá thang điểm 5 ở biến Rating
invalid_rating = ((df_rating_icons['Rating'] <= 0) | (df_rating_icons['Rating'] > 5)).sum()
print(f"Số lượng dòng có trường `Rating` nằm ngoài khoảng (1-5): {invalid_rating}")

# 2. Kiểm tra ký tự khoảng trắng thừa ở đầu/cuối của các chuỗi định tính nhằm tránh lỗi hiển thị
has_spaces_emoji = (df_rating_icons['Emoji'].astype(str).str.startswith(' ') | df_rating_icons['Emoji'].astype(str).str.endswith(' ')).sum()
has_spaces_star = (df_rating_icons['Star'].astype(str).str.startswith(' ') | df_rating_icons['Star'].astype(str).str.endswith(' ')).sum()

print(f"Số lượng dòng có trường `Emoji` bị dính khoảng trắng thừa ở đầu/cuối: {has_spaces_emoji}")
print(f"Số lượng dòng có trường `Star` bị dính khoảng trắng thừa ở đầu/cuối: {has_spaces_star}")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Số lượng dòng có trường `Rating` nằm ngoài khoảng (1-5): 0
Số lượng dòng có trường `Emoji` bị dính khoảng trắng thừa ở đầu/cuối: 0
Số lượng dòng có trường `Star` bị dính khoảng trắng thừa ở đầu/cuối: 0


Nhận xét về giá trị bất thường: 
1. Logic thang đo định danh: <br>
Trường `Rating` nhận các giá trị hoàn toàn hợp lệ, nằm trọn vẹn trong biên độ thang đo tiêu chuẩn từ 1 đến 5 sao, không có giá trị âm hoặc vượt ngưỡng rủi ro.

2. Tính toàn vẹn của chuỗi ký tự: <br>
Không phát hiện hiện tượng dính khoảng trắng thừa ở đầu/cuối tại hai biến `Emoji` và `Star`.

### 1.5 blinkit_orders.csv

Bộ dữ liệu `blinkit_orders.csv` là bảng dữ liệu giao dịch cốt lõi lưu trữ lịch sử biến động về dòng tiền hóa đơn và tiến trình kho vận cấp độ đơn hàng trên hệ thống Blinkit. Bảng này đóng vai trò trung tâm để kết nối với các bảng danh mục nhằm phục vụ công tác đánh giá hiệu suất vận chuyển và cấu trúc doanh thu.

Bộ dữ liệu gồm 10 biến như sau:
- Biến định danh và liên kết:<br>
`order_id`: Mã định danh duy nhất của từng đơn hàng (Khóa chính).<br>
`customer_id`: Mã định danh khách hàng thực hiện giao dịch (Khóa ngoại).<br>
`delivery_partner_id`: Mã định danh đối tác giao hàng đảm nhận vận chuyển (Khóa ngoại).<br>
`store_id`: Mã định danh kho tối xuất hàng (Khóa ngoại).

- Biến tiến trình và trạng thái vận chuyển:<br>
`order_date`: Thời điểm hệ thống ghi nhận đơn đặt hàng thành công.<br>
`promised_delivery_time`: Thời hạn cam kết giao hàng hiển thị trên ứng dụng người dùng.<br>
`actual_delivery_time`: Thời điểm thực tế đối tác giao hàng xác nhận hoàn thành.<br>
`delivery_status`: Trạng thái phân loại tiến độ giao hàng (On Time, Slightly Delayed, Significantly Delayed).

- Biến tài chính và phương thức giao dịch:<br>
`order_total`: Tổng giá trị tiền tệ của hóa đơn đơn hàng.<br>
`payment_method`: Phương thức thanh toán được áp dụng (Cash, UPI, Card, Wallet).

In [ ]:
# Đọc dữ liệu giao dịch thực tế từ file người dùng cung cấp
df_orders = pd.read_csv('blinkit_orders.csv')

# Kiểm tra quy mô dòng và cột
print("Quy mô bộ dữ liệu thực tế (Dòng, Cột):", df_orders.shape)

# Hiển thị cấu trúc các trường thông tin đầu tiên
df_orders.head()

Quy mô bộ dữ liệu thực tế (Dòng, Cột): (5000, 10)


,order_id,customer_id,order_date,promised_delivery_time,actual_delivery_time,delivery_status,order_total,payment_method,delivery_partner_id,store_id
0,1961864118,30065862,2024-07-17 08:34:01,2024-07-17 08:52:01,2024-07-17 08:47:01,On Time,3197.07,Cash,63230,4771
1,1549769649,9573071,2024-05-28 13:14:29,2024-05-28 13:25:29,2024-05-28 13:27:29,On Time,976.55,Cash,14983,7534
2,9185164487,45477575,2024-09-23 13:07:12,2024-09-23 13:25:12,2024-09-23 13:29:12,On Time,839.05,UPI,39859,9886
3,9644738826,88067569,2023-11-24 16:16:56,2023-11-24 16:34:56,2023-11-24 16:33:56,On Time,440.23,Card,61497,7917
4,5427684290,83298567,2023-11-20 05:00:39,2023-11-20 05:17:39,2023-11-20 05:18:39,On Time,2526.68,Cash,84315,2741


Bộ dữ liệu bao gồm 5000 bản ghi và 10 biến.

In [ ]:
# Xây dựng bảng tổng hợp kỹ thuật 3 cột dựa trên cấu trúc thực tế của file
df_orders_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_orders.dtypes.astype(str),
    'Số lượng giá trị Null': df_orders.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_orders.nunique()
})

# Hiển thị bảng thông số
df_orders_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
order_id,int64,0,5000
customer_id,int64,0,2172
order_date,object,0,5000
promised_delivery_time,object,0,4999
actual_delivery_time,object,0,5000
delivery_status,object,0,3
order_total,float64,0,4550
payment_method,object,0,4
delivery_partner_id,int64,0,5000
store_id,int64,0,5000


Nhận xét:
1. Độ đầy đủ: Toàn bộ 10 biến số đều hoàn thiện 100%, không ghi nhận giá trị khuyết thiếu (Null = 0) trên 5000 dòng dữ liệu.
2. Tính duy nhất: Trường `order_id` có đúng 5000 giá trị phân biệt, hoàn toàn trùng khớp với tổng số dòng, đủ điều kiện thiết lập làm khóa chính.
3. Tính tương quan phân phối: Biến `customer_id` có 2172 giá trị phân biệt, cho thấy hành vi tái mua sắm từ tệp khách hàng hiện tại. Các biến trạng thái vận chuyển và phương thức thanh toán nhận số lượng giá trị phân biệt giới hạn, phản ánh đúng cấu trúc danh mục cố định của hệ thống.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra tính toàn vẹn của biến định lượng bắt buộc dương
invalid_total = (df_orders['order_total'] <= 0).sum()
print(f"Biến `order_total` có giá trị <= 0: {invalid_total} bản ghi.")

# 2. Kiểm tra logic tiến trình thời gian tịnh tiến
order_dt = pd.to_datetime(df_orders['order_date'])
promised_dt = pd.to_datetime(df_orders['promised_delivery_time'])
actual_dt = pd.to_datetime(df_orders['actual_delivery_time'])

print(f"Biến `actual_delivery_time` trước `order_date`: {(actual_dt < order_dt).sum()} bản ghi.")
print(f"Biến `promised_delivery_time` trước `order_date`: {(promised_dt < order_dt).sum()} bản ghi.")

# 3. Kiểm tra logic phân loại dựa trên biên độ trễ hẹn (phút)
delay_minutes = (actual_dt - promised_dt).dt.total_seconds() / 60.0
df_orders['delay_minutes'] = delay_minutes

print("\n--- KIỂM TRA BIÊN ĐỘ TRỄ HẠN THEO TRẠNG THÁI GIAO HÀNG ---")
print(df_orders.groupby('delivery_status')['delay_minutes'].agg(['count', 'min', 'max']))

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `order_total` có giá trị <= 0: 0 bản ghi.
Biến `actual_delivery_time` trước `order_date`: 0 bản ghi.
Biến `promised_delivery_time` trước `order_date`: 0 bản ghi.

--- KIỂM TRA BIÊN ĐỘ TRỄ HẠN THEO TRẠNG THÁI GIAO HÀNG ---
                       count   min   max
delivery_status                         
On Time                 3470  -5.0   5.0
Significantly Delayed    493  16.0  30.0
Slightly Delayed        1037   6.0  15.0


Nhận xét về giá trị bất thường:
1. Tính toàn vẹn dòng tiền và tiến trình: Không phát hiện giá trị âm hoặc bằng 0 tại biến `order_total`. Tiến trình thời gian tuân thủ nghiêm ngặt trình tự nghiệp vụ logic tịnh tiến, không ghi nhận lỗi đảo ngược thời gian.
2. Quy tắc khoảng thời gian ân hạn vận hành: Hệ thống áp dụng quy tắc ân hạn cố định cho đối tác giao hàng: nhãn `On Time` áp dụng cho đơn hàng dao động từ -5.0 phút đến tối đa +5.0 phút so với thời gian cam kết; nhãn `Slightly Delayed` kích hoạt từ +6.0 đến +15.0 phút; nhãn `Significantly Delayed` áp dụng từ +16.0 đến +30.0 phút. Dữ liệu phân loại đồng nhất, không chứa lỗi xung đột gán nhãn.
3. Điểm bất thường cấu trúc hệ thống: Ghi nhận bất thường tại hai trường khóa ngoại `delivery_partner_id` và `store_id` khi cả hai đều đạt 5000 giá trị duy nhất trên tổng số 5000 bản ghi. Đặc điểm này chỉ ra dữ liệu đã bị sinh ngẫu nhiên hoặc được mã hóa ẩn danh độc lập theo từng dòng, triệt tiêu khả năng thực hiện các phép toán gộp nhóm theo kho hàng hoặc đối tác giao hàng trên bảng độc lập này.

In [ ]:
# Xuất bảng thống kê mô tả cho trường định lượng duy nhất của bảng gốc
df_orders[['order_total']].describe().round(2)

,order_total
count,5000.00
mean,2201.86
std,1303.02
min,13.25
25%,1086.21
50%,2100.69
75%,3156.88
max,6721.46


Nhận xét về các chỉ số thống kê:
1. Biến `order_total`: Giá trị trung bình đạt 2201.86, trung vị đạt 2100.69. Sự tiệm cận sát sao giữa hai chỉ số chứng minh phân phối của biến hội tụ về phân phối đối xứng, không bị lệch quá mức bởi các giá trị dị biệt thô.
2. Biên độ biến thiên: Độ lệch chuẩn đạt 1303.02, thể hiện dải biến thiên sức mua rộng nhưng ổn định xung quanh tâm dữ liệu. Khoảng tứ phân vị tập trung vững chắc từ 1086.21 (mức 25%) đến 3156.88 (mức 75%) trên một lượt giao dịch. Giá trị nhỏ nhất ghi nhận ở mức 13.25 và giá trị lớn nhất đạt ngưỡng 6721.46.

### 1.6 blinkit_order_items.csv

Bộ dữ liệu `blinkit_order_items.csv` là bảng giao dịch chi tiết cấp độ mục hàng. Bảng này đóng vai trò trung gian giải quyết mối quan hệ nhiều-nhiều giữa đơn hàng và danh mục sản phẩm, lưu trữ cơ cấu cấu thành của từng giao dịch.

Bộ dữ liệu gồm 4 biến như sau:
- Biến định danh và liên kết:<br>
`order_id`: Mã định danh đơn hàng (Khóa ngoại kết nối với bảng đơn hàng).<br>
`product_id`: Mã định danh sản phẩm (Khóa ngoại kết nối với bảng sản phẩm).

- Biến định lượng:<br>
`quantity`: Số lượng sản phẩm được đặt mua trên một dòng mặt hàng.<br>
`unit_price`: Đơn giá bán thực tế của một đơn vị sản phẩm tại thời điểm giao dịch, tương ứng với biến `price` ở bảng `blinkit_products.csv`.

In [ ]:
# Đọc dữ liệu chi tiết mục hàng
df_order_items = pd.read_csv('blinkit_order_items.csv')

# Kiểm tra quy mô ma trận dữ liệu
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_order_items.shape)

# Hiển thị cấu trúc các bản ghi đầu tiên
df_order_items.head()

Quy mô bộ dữ liệu (Dòng, Cột): (5000, 4)


,order_id,product_id,quantity,unit_price
0,1961864118,642612,3,517.03
1,1549769649,378676,1,881.42
2,9185164487,741341,2,923.84
3,9644738826,561860,1,874.78
4,5427684290,602241,2,976.55


Bộ dữ liệu bao gồm 5000 bản ghi và 4 biến.

In [ ]:
# Xây dựng bảng tổng hợp thông số kỹ thuật nền tảng
df_items_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_order_items.dtypes.astype(str),
    'Số lượng giá trị Null': df_order_items.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_order_items.nunique()
})

# Hiển thị bảng tổng hợp
df_items_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
order_id,int64,0,5000
product_id,int64,0,268
quantity,int64,0,3
unit_price,float64,0,267


Nhận xét:
1. Độ đầy đủ: <br>
Toàn bộ 4 biến đều hoàn thiện, không ghi nhận giá trị khuyết thiếu (Null = 0) trên 5000 dòng dữ liệu.

2. Tính duy nhất: <br>
Trường `order_id` ghi nhận đúng 5000 giá trị phân biệt, trùng khớp hoàn toàn với tổng số dòng dữ liệu, cho thấy mỗi đơn hàng hiện chỉ chứa duy nhất một dòng sản phẩm.

3. Tính tương quan: <br>
Trường `product_id` ghi nhận 268 giá trị phân biệt, trùng khớp hoàn toàn với số lượng mã tại danh mục sản phẩm, đảm bảo tính toàn vẹn tham chiếu hệ thống.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra giá trị <= 0 ở các biến định lượng bắt buộc dương
print(f"Biến `quantity` có giá trị <= 0: {(df_order_items['quantity'] <= 0).sum()} bản ghi.")
print(f"Biến `unit_price` có giá trị <= 0: {(df_order_items['unit_price'] <= 0).sum()} bản ghi.")

# 2. Kiểm tra tính trùng lặp khóa tổ hợp (order_id, product_id)
duplicate_combinations = df_order_items.duplicated(subset=['order_id', 'product_id']).sum()
print(f"Tổ hợp (order_id, product_id) bị lặp lại: {duplicate_combinations} bản ghi.")

# 3. Kiểm tra logic dòng tiền chéo với bảng đơn hàng (blinkit_orders.csv)
df_orders = pd.read_csv('blinkit_orders.csv')
df_merged = pd.merge(df_orders, df_order_items, on='order_id', how='inner')
df_merged['calc_item_total'] = df_merged['quantity'] * df_merged['unit_price']
correlation = df_merged['order_total'].corr(df_merged['calc_item_total'])
print(f"Hệ số tương quan giữa `order_total` và tích (`quantity` * `unit_price`): {correlation:.6f}")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `quantity` có giá trị <= 0: 0 bản ghi.
Biến `unit_price` có giá trị <= 0: 0 bản ghi.
Tổ hợp (order_id, product_id) bị lặp lại: 0 bản ghi.


Hệ số tương quan giữa `order_total` và tích (`quantity` * `unit_price`): -0.008442


Nhận xét về giá trị bất thường:
1. Tính toàn vẹn giá trị số: <br>
Không phát hiện giá trị âm hoặc bằng 0 tại `quantity` và `unit_price`. Tổ hợp khóa liên kết `order_id` và `product_id` đạt trạng thái duy nhất tuyệt đối, không ghi nhận lỗi lặp dòng cấu trúc.

2. Mâu thuẫn logic tài chính chéo: <br>
Hệ số tương quan giữa `order_total` và tích số lý thuyết gần như bằng 0 (-0.008442). Về mặt nghiệp vụ kế toán thương mại, tổng giá trị hóa đơn bắt buộc phải tỷ lệ thuận chặt chẽ với tổng giá trị mục hàng thành phần. Việc độc lập tuyến tính hoàn toàn chỉ ra dữ liệu của hai bảng giao dịch đã bị tạo ngẫu nhiên, độc lập hoặc xảy ra lỗi ngắt kết nối tích hợp trong quy trình trích xuất hệ thống.

In [ ]:
# Xuất bảng thống kê mô tả cho các trường định lượng thực sự
df_order_items[['quantity', 'unit_price']].describe().round(2)

,quantity,unit_price
count,5000.00,5000.00
mean,2.01,493.16
std,0.82,298.08
min,1.00,12.32
25%,1.00,227.22
50%,2.00,448.16
75%,3.00,781.08
max,3.00,995.98


Nhận xét về các chỉ số thống kê:
1. Biến `quantity`: <br>
Nhận các giá trị nguyên rời rạc từ 1.00 đến 3.00 đơn vị sản phẩm trên mỗi dòng hóa đơn. Giá trị trung bình đạt 2.01, trung vị đạt 2.00, thể hiện phân phối đối xứng cao và phản ánh quy mô giỏ hàng đơn lẻ khối lượng nhỏ đặc trưng của mô hình giao hàng siêu tốc.

2. Biến `unit_price`: <br>
Giá trị trung bình đạt 493.16, trung vị đạt 448.16. Độ lệch chuẩn đạt 298.08 cho thấy dải phân tán rộng. Khoảng tứ phân vị tập trung đều từ 227.22 (mức 25%) đến 781.08 (mức 75%), chứng minh cơ cấu hàng hóa phát sinh giao dịch dàn trải đều trên các phân khúc giá từ thấp nhất 12.32 đến cao nhất 995.98.

### 1.7 blinkit_customer_feedback.csv

Bộ dữ liệu `blinkit_customer_feedback.csv` là bảng dữ liệu giao dịch ghi nhận ý kiến, điểm số đánh giá và trạng thái cảm xúc của người tiêu dùng sau khi hoàn thành đơn hàng. Bảng này cung cấp cơ sở dữ liệu định tính và định lượng phục vụ công tác kiểm định chất lượng dịch vụ và sản phẩm.

Bộ dữ liệu gồm 8 biến như sau:
- Biến định danh và liên kết:<br>
`feedback_id`: Mã định danh duy nhất của từng phản hồi (Khóa chính).<br>
`order_id`: Mã định danh đơn hàng liên quan (Khóa ngoại kết nối với bảng đơn hàng).<br>
`customer_id`: Mã định danh khách hàng thực hiện đánh giá (Khóa ngoại).

- Biến định tính và trạng thái:<br>
`feedback_text`: Nội dung văn bản ý kiến phản hồi của khách hàng.<br>
`feedback_category`: Danh mục phân loại khía cạnh phản hồi (Delivery, App Experience, Customer Service, Product Quality).<br>
`sentiment`: Phân loại trạng thái ý kiến (Negative, Neutral, Positive).<br>
`feedback_date`: Ngày ghi nhận phản hồi trên hệ thống.

- Biến định lượng:<br>
`rating`: Điểm số đánh giá mức độ hài lòng bằng số nguyên từ 1 đến 5.

In [ ]:
# Đọc dữ liệu phản hồi khách hàng
df_feedback = pd.read_csv('blinkit_customer_feedback.csv')

# Kiểm tra quy mô ma trận dữ liệu
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_feedback.shape)

# Hiển thị cấu trúc các bản ghi đầu tiên
df_feedback.head()

Quy mô bộ dữ liệu (Dòng, Cột): (5000, 8)


,feedback_id,order_id,customer_id,rating,feedback_text,feedback_category,sentiment,feedback_date
0,2234710,1961864118,30065862,4,"It was okay, nothing special.",Delivery,Neutral,2024-07-17
1,5450964,1549769649,9573071,3,The order was incorrect.,App Experience,Negative,2024-05-28
2,482108,9185164487,45477575,3,"It was okay, nothing special.",App Experience,Neutral,2024-09-23
3,4823104,9644738826,88067569,4,The product met my expectations.,App Experience,Neutral,2023-11-24
4,3537464,5427684290,83298567,3,Product was damaged during delivery.,Delivery,Negative,2023-11-20


Bộ dữ liệu bao gồm 5000 bản ghi và 8 biến.

In [ ]:
# Xây dựng bảng tổng hợp thông số kỹ thuật nền tảng
df_feedback_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_feedback.dtypes.astype(str),
    'Số lượng giá trị Null': df_feedback.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_feedback.nunique()
})

# Hiển thị bảng tổng hợp
df_feedback_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
feedback_id,int64,0,5000
order_id,int64,0,5000
customer_id,int64,0,2172
rating,int64,0,5
feedback_text,object,0,25
feedback_category,object,0,4
sentiment,object,0,3
feedback_date,object,0,600


Nhận xét:
1. Độ đầy đủ: <br>
Toàn bộ 8 biến số đều đạt tỷ lệ hoàn thiện 100%, không ghi nhận giá trị khuyết thiếu (Null = 0) trên 5000 dòng dữ liệu.

2. Tính duy nhất: <br>
Trường `feedback_id` có đúng 5000 giá trị phân biệt, hoàn toàn trùng khớp với tổng số dòng, đủ điều kiện thiết làm làm khóa chính. Trường `order_id` có đúng 5000 giá trị duy nhất, cho thấy mỗi đơn hàng trong hệ thống hiện chỉ nhận tối đa một lượt phản hồi.

3. Tính tương quan phân phối: <br>
Trường `customer_id` có 2172 giá trị phân biệt, nghĩa là 100% các khách hàng đều feedback lại. Biến `feedback_text` nhận 25 chuỗi văn bản mẫu cố định, thể hiện dữ liệu được thu thập qua danh sách câu chọn sẵn thay vì văn bản tự do.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra biên độ giá trị của biến định lượng bắt buộc
invalid_rating = ((df_feedback['rating'] < 1) | (df_feedback['rating'] > 5)).sum()
print(f"Biến `rating` nằm ngoài dải (1-5): {invalid_rating} bản ghi.")

# 2. Thống kê chéo kiểm tra tính nhất quán giữa điểm số và trạng thái cảm xúc
print("\n--- THỐNG KÊ CHÉO GIỮA ĐIỂM SỐ VÀ TRẠNG THÁI CẢM XÚC ---")
print(pd.crosstab(df_feedback['rating'], df_feedback['sentiment']))

# 3. Kiểm tra logic tiến trình thời gian so với bảng đơn hàng
df_orders = pd.read_csv('blinkit_orders.csv')
df_merged = pd.merge(df_orders, df_feedback, on='order_id', how='inner')
order_days = pd.to_datetime(df_merged['order_date']).dt.date
feedback_days = pd.to_datetime(df_merged['feedback_date']).dt.date
invalid_dates = (feedback_days < order_days).sum()
print(f"\nSố lượng bản ghi có ngày phản hồi trước ngày đặt hàng: {invalid_dates}")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `rating` nằm ngoài dải (1-5): 0 bản ghi.

--- THỐNG KÊ CHÉO GIỮA ĐIỂM SỐ VÀ TRẠNG THÁI CẢM XÚC ---
sentiment  Negative  Neutral  Positive
rating                                
1               540        0         0
2               538        0         0
3               564      834         0
4                 0      904       804
5                 0        0       816



Số lượng bản ghi có ngày phản hồi trước ngày đặt hàng: 0


Nhận xét về giá trị bất thường:
1. Tính toàn vẹn của thang đo và tiến trình: <br>
Biến `rating` tuân thủ dải điểm số nguyên từ 1 đến 5. Tiến trình thời gian đồng nhất, ngày phản hồi `feedback_date` luôn xảy ra bằng hoặc sau ngày đặt hàng, trong đó 98.72% phản hồi (4936 bản ghi) được ghi nhận ngay trong ngày đặt hàng.

2. Tính nhất quán logic nghiệp vụ: <br>
Ma trận thống kê chéo giữa `rating` và `sentiment` thể hiện tính nhất quán phân loại cao: điểm số 1 và 2 liên kết hoàn toàn với trạng thái tiêu cực; điểm số 5 liên kết hoàn toàn với trạng thái tích cực; điểm số 3 và 4 phân phối hợp lý giữa trạng thái trung lập và các trạng thái biên, không ghi nhận lỗi xung đột nghịch lý dữ liệu.

In [ ]:
# Xuất bảng thống kê mô tả cho trường định lượng điểm số
df_feedback[['rating']].describe().round(2)

,rating
count,5000.00
mean,3.34
std,1.19
min,1.00
25%,3.00
50%,4.00
75%,4.00
max,5.00


Nhận xét về các chỉ số thống kê:
1. Biến `rating`: <br>
Giá trị trung bình đạt 3.34, trung vị đạt 4.00, thể hiện mức độ đánh giá tổng thể của người dùng ở mức khá. Khoảng tứ phân vị tập trung từ 3.00 (mức 25%) đến 4.00 (mức 75%), cho thấy hơn 50% lượng phản hồi tập trung vào nhóm điểm số trung bình và khá.

2. Đặc trưng phân tán: <br>
Đọc lập dải biến thiên ghi nhận độ lệch chuẩn đạt 1.19, chứng minh dải điểm số có mức độ phân tán vừa phải quanh tâm dữ liệu, phản ánh sự phân hóa rõ rệt trong trải nghiệm dịch vụ của tệp khách hàng.

### 1.8 blinkit_delivery_performance.csv

Bộ dữ liệu `blinkit_delivery_performance.csv` là bảng dữ liệu giao dịch ghi nhận chi tiết hiệu suất kho vận và các chỉ số đo lường hành trình giao hàng của đối tác tài xế. Bảng này kết nối trực tiếp với hệ thống đơn hàng nhằm đánh giá tốc độ xử lý logistics và phân tích nguyên nhân gây trễ hẹn.

Bộ dữ liệu gồm 8 biến như sau:
- Biến định danh và liên kết:<br>
`order_id`: Mã định danh đơn hàng (Khóa chính).<br>
`delivery_partner_id`: Mã định danh đối tác giao hàng (Khóa ngoại).

- Biến tiến trình và trạng thái vận chuyển:<br>
`promised_time`: Thời hạn cam kết giao hàng.<br>
`actual_time`: Thời điểm thực tế hoàn thành giao hàng.<br>
`delivery_status`: Trạng thái phân loại tiến độ giao hàng (On Time, Slightly Delayed, Significantly Delayed).<br>
`reasons_if_delayed`: Nguyên nhân chậm trễ đơn hàng (Traffic).

- Biến định lượng:<br>
`delivery_time_minutes`: Biên độ chênh lệch thời gian giao hàng tính theo phút.<br>
`distance_km`: Khoảng cách vận chuyển thực tế tính theo ki-lô-mét.

In [ ]:
# Đọc dữ liệu hiệu suất giao hàng thực tế
df_delivery_perf = pd.read_csv('blinkit_delivery_performance.csv')

# Kiểm tra quy mô ma trận dữ liệu
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_delivery_perf.shape)

# Hiển thị cấu trúc các bản ghi đầu tiên
df_delivery_perf.head()

Quy mô bộ dữ liệu (Dòng, Cột): (5000, 8)


,order_id,delivery_partner_id,promised_time,actual_time,delivery_time_minutes,distance_km,delivery_status,reasons_if_delayed
0,1961864118,63230,2024-07-17 08:52:01,2024-07-17 08:47:01,-5.0,0.96,On Time,NaN
1,1549769649,14983,2024-05-28 13:25:29,2024-05-28 13:27:29,2.0,0.98,On Time,Traffic
2,9185164487,39859,2024-09-23 13:25:12,2024-09-23 13:29:12,4.0,3.83,On Time,Traffic
3,9644738826,61497,2023-11-24 16:34:56,2023-11-24 16:33:56,-1.0,2.76,On Time,NaN
4,5427684290,84315,2023-11-20 05:17:39,2023-11-20 05:18:39,1.0,2.63,On Time,Traffic


Bộ dữ liệu bao gồm 5000 bản ghi và 8 biến.

In [ ]:
# Xây dựng bảng tổng hợp thông số kỹ thuật nền tảng
df_perf_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_delivery_perf.dtypes.astype(str),
    'Số lượng giá trị Null': df_delivery_perf.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_delivery_perf.nunique()
})

# Hiển thị bảng tổng hợp
df_perf_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
order_id,int64,0,5000
delivery_partner_id,int64,0,5000
promised_time,object,0,4999
actual_time,object,0,5000
delivery_time_minutes,float64,0,36
distance_km,float64,0,451
delivery_status,object,0,3
reasons_if_delayed,object,1902,1


Nhận xét:
1. Độ đầy đủ: Biến `reasons_if_delayed` khuyết thiếu 1902 giá trị (chiếm 38.04%). Toàn bộ 7 biến số còn lại đạt tỷ lệ hoàn thiện 100% trên 5000 dòng dữ liệu.
2. Tính duy nhất: Trường `order_id` có đúng 5000 giá trị phân biệt, hoàn toàn trùng khớp với tổng số dòng, đủ điều kiện thiết lập làm khóa chính.
3. Tính tương quan phân phối: Biến `delivery_partner_id` có đúng 5000 giá trị duy nhất, thể hiện đặc tính dữ liệu bị sinh ngẫu nhiên hoặc ẩn danh theo từng bản ghi tương tự như bảng đơn hàng tổng hợp. Biến `delivery_status` có 3 trạng thái phân loại và `reasons_if_delayed` chỉ chứa 1 nhãn duy nhất.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra tính toàn vẹn của biến định lượng bắt buộc dương và logic thời gian
print(f"Biến `distance_km` có giá trị <= 0: {(df_delivery_perf['distance_km'] <= 0).sum()} bản ghi.")

calc_diff = (pd.to_datetime(df_delivery_perf['actual_time']) - pd.to_datetime(df_delivery_perf['promised_time'])).dt.total_seconds() / 60.0
time_mismatch = (df_delivery_perf['delivery_time_minutes'] - calc_diff).abs().gt(0.1).sum()
print(f"Số lượng dòng sai lệch giữa `delivery_time_minutes` và hiệu số thời gian thực tế: {time_mismatch} bản ghi.")

# 2. Kiểm tra xung đột logic giữa trạng thái vận chuyển và nguyên nhân chậm trễ
print("\n--- THỐNG KÊ CHÉO GIỮA TRẠNG THÁI VÀ NGUYÊN NHÂN CHẬM TRỄ ---")
print(df_delivery_perf.groupby('delivery_status')['reasons_if_delayed'].value_counts(dropna=False))

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `distance_km` có giá trị <= 0: 0 bản ghi.
Số lượng dòng sai lệch giữa `delivery_time_minutes` và hiệu số thời gian thực tế: 0 bản ghi.

--- THỐNG KÊ CHÉO GIỮA TRẠNG THÁI VÀ NGUYÊN NHÂN CHẬM TRỄ ---
delivery_status        reasons_if_delayed
On Time                NaN                   1902
                       Traffic               1568
Significantly Delayed  Traffic                493
Slightly Delayed       Traffic               1037
Name: count, dtype: int64


Nhận xét về giá trị bất thường:
1. Tính toàn vẹn tiến trình và khoảng cách: Không phát hiện giá trị âm hoặc bằng 0 tại biến `distance_km`. Trường `delivery_time_minutes` khớp chính xác tuyệt đối với hiệu số thời gian tính toán giữa `actual_time` và `promised_time`.
2. Mâu thuẫn logic phân loại và nguyên nhân: Ghi nhận 1568 bản ghi thuộc trạng thái đúng hẹn `On Time` nhưng vẫn bị gán nguyên nhân chậm trễ là `Traffic`. Các đơn hàng hoàn thành đúng hạn không cần thiết phải gán lý do trễ, điều này phản ánh lỗi gán tự động của hệ thống trích xuất. Toàn bộ các đơn hàng bị trễ (`Slightly Delayed` và `Significantly Delayed`) đều được điền lý do `Traffic` hoàn chỉnh, không bị bỏ sót.

In [ ]:
# Xuất bảng thống kê mô tả cho các trường định lượng thực sự
df_delivery_perf[['delivery_time_minutes', 'distance_km']].describe().round(2)

,delivery_time_minutes,distance_km
count,5000.00,5000.00
mean,4.44,2.72
std,8.06,1.29
min,-5.00,0.50
25%,-1.00,1.59
50%,2.00,2.69
75%,8.00,3.85
max,30.00,5.00


Nhận xét về các chỉ số thống kê:
1. Biến `delivery_time_minutes`: Giá trị trung bình đạt 4.44 phút, trung vị đạt 2.00 phút. Phân phối lệch phải với biên độ dao động từ -5.00 phút đến tối đa +30.00 phút. Khoảng tứ phân vị tập trung từ -1.00 phút đến +8.00 phút, phản ánh quy mô thời gian xử lý nhanh và tính hiệu quả của khoảng thời gian ân hạn.
2. Biến `distance_km`: Giá trị trung bình đạt 2.72 km, trung vị đạt 2.69 km, độ lệch chuẩn đạt 1.29 km. Khoảng cách vận chuyển ngắn nhất là 0.50 km và dài nhất là 5.00 km, cho thấy phạm vi phục vụ tập trung trong bán kính hẹp đặc trưng của mô hình phân phối siêu tốc. Khoảng tứ phân vị tập trung ổn định từ 1.59 km đến 3.85 km.

### 1.9 blinkit_marketing_performance.csv

Bộ dữ liệu `blinkit_marketing_performance.csv` là bảng dữ liệu giao dịch lưu trữ các chỉ số đo lường hiệu suất và chi phí của các chiến dịch quảng cáo, tiếp thị trên hệ thống Blinkit. Bảng dữ liệu này phục vụ công tác đánh giá hiệu quả ngân sách và khả năng thu hút người dùng của từng kênh truyền thông.

Bộ dữ liệu gồm 11 biến như sau:
- Biến định danh và phân loại:<br>
`campaign_id`: Mã định danh duy nhất của từng chiến dịch tiếp thị (Khóa chính).<br>
`campaign_name`: Tên phân loại chiến dịch tiếp thị.<br>
`target_audience`: Nhóm đối tượng khách hàng mục tiêu của chiến dịch.<br>
`channel`: Kênh truyền thông triển khai tiếp thị.

- Biến tiến trình và thời gian:<br>
`date`: Ngày ghi nhận các chỉ số hiệu suất của chiến dịch quảng cáo.

- Biến định lượng đo lường:<br>
`impressions`: Số lượt hiển thị của chiến dịch quảng cáo.<br>
`clicks`: Số lượt nhấn vào nội dung chiến dịch quảng cáo.<br>
`conversions`: Số lượng lượt chuyển đổi hành vi thành công.<br>
`spend`: Ngân sách chi phí đã chi trả cho chiến dịch.<br>
`revenue_generated`: Doanh thu thu về phát sinh từ chiến dịch.<br>
`roas`: Tỷ lệ doanh thu thu về trên chi phí ngân sách tiếp thị.

In [ ]:
# Đọc dữ liệu hiệu suất tiếp thị
df_marketing = pd.read_csv('blinkit_marketing_performance.csv')

# Kiểm tra quy mô ma trận dữ liệu
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_marketing.shape)

# Hiển thị cấu trúc các bản ghi đầu tiên
df_marketing.head()

Quy mô bộ dữ liệu (Dòng, Cột): (5400, 11)


,campaign_id,campaign_name,date,target_audience,channel,impressions,clicks,conversions,spend,revenue_generated,roas
0,548299,New User Discount,2024-11-05,Premium,App,3130,163,78,1431.85,4777.75,3.60
1,390914,Weekend Special,2024-11-05,Inactive,App,3925,494,45,4506.34,6238.11,2.98
2,834385,Festival Offer,2024-11-05,Inactive,Email,7012,370,78,4524.23,2621.00,2.95
3,241523,Flash Sale,2024-11-05,Inactive,SMS,1115,579,86,3622.79,2955.00,2.84
4,595111,Membership Drive,2024-11-05,New Users,Email,7172,795,54,2888.99,8951.81,2.22


Bộ dữ liệu bao gồm 5400 bản ghi và 11 biến.

In [ ]:
# Xây dựng bảng tổng hợp thông số kỹ thuật nền tảng
df_mkt_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_marketing.dtypes.astype(str),
    'Số lượng giá trị Null': df_marketing.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_marketing.nunique()
})

# Hiển thị bảng tổng hợp
df_mkt_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
campaign_id,int64,0,5400
campaign_name,object,0,9
date,object,0,600
target_audience,object,0,4
channel,object,0,4
impressions,int64,0,4058
clicks,int64,0,899
conversions,int64,0,91
spend,float64,0,5356
revenue_generated,float64,0,5386


Nhận xét:
1. Độ đầy đủ: <br>
Toàn bộ 11 biến số đều đạt tỷ lệ hoàn thiện 100%, không ghi nhận giá trị khuyết thiếu (Null = 0) trên 5400 dòng dữ liệu.

2. Tính duy nhất: <br>
Trường `campaign_id` ghi nhận đúng 5400 giá trị phân biệt, trùng khớp hoàn toàn với tổng số dòng dữ liệu, đủ điều kiện thiết lập làm khóa chính.

3. Đặc điểm phân phối phân loại: <br>
Trường `campaign_name` chứa 9 tên chiến dịch cố định; `target_audience` gồm 4 nhóm đối tượng khách hàng; `channel` gồm 4 kênh truyền thông chính.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra tính toàn vẹn của các biến định lượng bắt buộc dương
check_neg_cols = ['impressions', 'clicks', 'conversions', 'spend', 'revenue_generated', 'roas']
for col in check_neg_cols:
    print(f"Biến `{col}` có giá trị < 0: {(df_marketing[col] < 0).sum()} bản ghi.")

# 2. Kiểm tra logic phễu chuyển đổi: impressions >= clicks >= conversions
invalid_funnel_clicks = (df_marketing['clicks'] > df_marketing['impressions']).sum()
invalid_funnel_conv = (df_marketing['conversions'] > df_marketing['clicks']).sum()
print(f"Số lượng bản ghi lỗi phễu tiếp thị (clicks > impressions): {invalid_funnel_clicks}")
print(f"Số lượng bản ghi lỗi phễu tiếp thị (conversions > clicks): {invalid_funnel_conv}")

# 3. Kiểm tra logic nội bộ biến định lượng: roas == revenue_generated / spend
calc_roas = df_marketing['revenue_generated'] / df_marketing['spend']
mismatch_roas = (df_marketing['roas'] - calc_roas).abs().gt(0.05).sum()
print(f"Số lượng bản ghi sai lệch logic tỷ lệ hiệu quả tiếp thị: {mismatch_roas}")

# Đo lường hệ số tương quan tuyến tính giữa roas ghi nhận và tỷ lệ tính toán thực tế
correlation = df_marketing['roas'].corr(calc_roas)
print(f"Hệ số tương quan tuyến tính giữa hai trường số liệu: {correlation:.6f}")

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `impressions` có giá trị < 0: 0 bản ghi.
Biến `clicks` có giá trị < 0: 0 bản ghi.
Biến `conversions` có giá trị < 0: 0 bản ghi.
Biến `spend` có giá trị < 0: 0 bản ghi.
Biến `revenue_generated` có giá trị < 0: 0 bản ghi.
Biến `roas` có giá trị < 0: 0 bản ghi.
Số lượng bản ghi lỗi phễu tiếp thị (clicks > impressions): 0
Số lượng bản ghi lỗi phễu tiếp thị (conversions > clicks): 0
Số lượng bản ghi sai lệch logic tỷ lệ hiệu quả tiếp thị: 5271
Hệ số tương quan tuyến tính giữa hai trường số liệu: -0.000110


Nhận xét về giá trị bất thường:
1. Tính toàn vẹn của phễu tiếp thị: <br>
Không phát hiện giá trị âm tại các biến định lượng đo lường. Logic cấu trúc phễu quảng cáo đạt trạng thái hợp lệ, số lượt hiển thị luôn lớn hơn hoặc bằng số lượt nhấn, và số lượt nhấn luôn lớn hơn hoặc bằng số lượng chuyển đổi thành công trên tất cả các bản ghi.

2. Mâu thuẫn logic hiệu số nội bộ: <br>
Ghi nhận 5271 bản ghi (chiếm 97.61%) có giá trị trường `roas` sai lệch hoàn toàn so với tỷ lệ toán học lý thuyết được tính toán trực tiếp từ công thức doanh thu chia cho chi phí quảng cáo (`revenue_generated / spend`). Hệ số tương quan tuyến tính giữa hai chuỗi số liệu này gần như bằng 0 (-0.000110). Đặc điểm này khẳng định các biến số tài chính quảng cáo của bảng dữ liệu đã bị tạo ngẫu nhiên độc lập, triệt tiêu tính nhất quán nội bộ của thực thể chiến dịch.

In [ ]:
# Xuất bảng thống kê mô tả cho các trường định lượng hiệu suất tiếp thị
mkt_num_cols = ['impressions', 'clicks', 'conversions', 'spend', 'revenue_generated', 'roas']
df_marketing[mkt_num_cols].describe().round(2)

,impressions,clicks,conversions,spend,revenue_generated,roas
count,5400.00,5400.00,5400.00,5400.00,5400.00,5400.00
mean,5460.67,550.77,55.19,3022.19,5961.74,2.74
std,2571.78,260.08,26.15,1148.73,2322.24,0.72
min,1002.00,100.00,10.00,1000.63,2003.10,1.50
25%,3231.50,322.00,32.00,2029.07,3907.24,2.12
50%,5457.50,555.00,55.00,3042.48,5935.94,2.72
75%,7676.25,772.00,78.00,4011.57,7973.71,3.37
max,9999.00,1000.00,100.00,4997.55,9999.54,4.00


Nhận xét về các chỉ số thống kê:
1. Biến đo lường quy mô (`impressions`, `clicks`, `conversions`): <br>
Mức độ hiển thị trung bình đạt 5460.67 lượt, lượt nhấn đạt 550.77 và lượt chuyển đổi đạt 55.19 trên mỗi chiến dịch. Tỷ lệ chuyển đổi trung bình của phễu tiếp thị đạt trạng thái ổn định với các giá trị trung vị nằm sát giá trị trung bình, phản ánh phân phối đối xứng và không có giá trị dị biệt cực đoan.

2. Biến số tài chính (`spend`, `revenue_generated`, `roas`): <br>
Chi phí ngân sách trung bình cho một chiến dịch đạt $3022.19, dải chi tiêu dao động ổn định từ $1000.63 đến $4997.55. Doanh thu trung bình tạo ra đạt $5961.74, phân vị tối đa đạt $9999.54. Trường `roas` ghi nhận dải phân phối tịnh tiến đều từ ngưỡng tối thiểu 1.50 đến ngưỡng tối đa 4.00, độ lệch chuẩn đạt 0.72 xung quanh tâm giá trị trung bình 2.74.

### 1.10 blinkit_inventory.csv

Bộ dữ liệu `blinkit_inventory.csv` là bảng dữ liệu giao dịch lưu trữ nhật ký biến động kho hàng theo ngày của từng mã sản phẩm trên hệ thống Blinkit. Bảng dữ liệu này phục vụ công tác theo dõi tiến độ nhập kho, quản trị tỷ lệ hao hụt và kiểm soát chất lượng chuỗi cung ứng.

Bộ dữ liệu gồm 4 biến như sau:
- Biến định danh và thời gian:<br>
`product_id`: Mã định danh sản phẩm (Khóa ngoại).<br>
`date`: Ngày ghi nhận trạng thái biến động của kho hàng.

- Biến định lượng kiểm soát kho:<br>
`stock_received`: Số lượng sản phẩm được nhập kho thành công.<br>
`damaged_stock`: Số lượng sản phẩm ghi nhận bị hư hỏng hoặc lỗi chất lượng.

In [ ]:
# Đọc dữ liệu nhật ký kho hàng
df_inventory = pd.read_csv('blinkit_inventory.csv')

# Kiểm tra quy mô ma trận dữ liệu
print("Quy mô bộ dữ liệu (Dòng, Cột):", df_inventory.shape)

# Hiển thị cấu trúc các bản ghi đầu tiên
df_inventory.head()

Quy mô bộ dữ liệu (Dòng, Cột): (75172, 4)


,product_id,date,stock_received,damaged_stock
0,153019,17-03-2023,4,2
1,848226,17-03-2023,4,2
2,965755,17-03-2023,1,0
3,39154,17-03-2023,4,0
4,34186,17-03-2023,3,2


Bộ dữ liệu bao gồm 75172 bản ghi và 4 biến.

In [ ]:
# Xây dựng bảng tổng hợp thông số kỹ thuật nền tảng
df_inv_quality = pd.DataFrame({
    'Kiểu dữ liệu': df_inventory.dtypes.astype(str),
    'Số lượng giá trị Null': df_inventory.isnull().sum(),
    'Số lượng giá trị phân biệt (Unique)': df_inventory.nunique()
})

# Hiển thị bảng tổng hợp
df_inv_quality

,Kiểu dữ liệu,Số lượng giá trị Null,Số lượng giá trị phân biệt (Unique)
product_id,int64,0,268
date,object,0,600
stock_received,int64,0,4
damaged_stock,int64,0,2


Nhận xét:
1. Độ đầy đủ: <br>
Toàn bộ 4 biến số đều đạt tỷ lệ hoàn thiện 100%, không ghi nhận giá trị khuyết thiếu (Null = 0) trên 75,172 dòng dữ liệu.

2. Tính duy nhất và cấu trúc khóa: <br>
Bảng không có trường mã định danh đơn lẻ làm khóa chính. Tổ hợp hai trường (`product_id`, `date`) ghi nhận 0 trường hợp trùng lặp, xác định đây là tập hợp khóa phức hợp đại diện cho duy nhất một trạng thái sản phẩm trong một ngày.

3. Đặc điểm phân phối phân loại: <br>
Trường `product_id` ghi nhận 268 mã sản phẩm phân biệt, đồng bộ với danh mục sản phẩm gốc. Trường `date` ghi nhận 600 ngày phân biệt, tương thích với dải chuỗi thời gian của các phân hệ giao dịch trước. Biến `stock_received` nhận 4 giá trị rời rạc và `damaged_stock` chỉ nhận 2 giá trị rời rạc.

In [ ]:
print("--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---")

# 1. Kiểm tra tính toàn vẹn của các biến định lượng bắt buộc không âm
print(f"Biến `stock_received` có giá trị < 0: {(df_inventory['stock_received'] < 0).sum()} bản ghi.")
print(f"Biến `damaged_stock` có giá trị < 0: {(df_inventory['damaged_stock'] < 0).sum()} bản ghi.")

# 2. Kiểm tra logic ràng buộc nội bộ kho hàng: damaged_stock <= stock_received
invalid_damage = (df_inventory['damaged_stock'] > df_inventory['stock_received']).sum()
print(f"Số lượng bản ghi lỗi logic kho hàng (damaged_stock > stock_received): {invalid_damage} bản ghi.")

# 3. Thống kê chi tiết ma trận tổ hợp giữa hai biến số kho hàng
print("\n--- MA TRẬN ĐỐI CHIẾU SỐ LƯỢNG NHẬP VÀ SỐ LƯỢNG HƯ HỎNG ---")
print(pd.crosstab(df_inventory['stock_received'], df_inventory['damaged_stock']))

--- KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG ---
Biến `stock_received` có giá trị < 0: 0 bản ghi.
Biến `damaged_stock` có giá trị < 0: 0 bản ghi.
Số lượng bản ghi lỗi logic kho hàng (damaged_stock > stock_received): 29255 bản ghi.

--- MA TRẬN ĐỐI CHIẾU SỐ LƯỢNG NHẬP VÀ SỐ LƯỢNG HƯ HỎNG ---
damaged_stock       0      2
stock_received              
0                   0  28411
1                2513    844
3               22008   7439
4               10517   3440


Nhận xét về giá trị bất thường:
1. Tính toàn vẹn của giá trị số: <br>
Không phát hiện giá trị âm tại các biến định lượng kiểm soát kho hàng.

2. Mâu thuẫn logic nghiệp vụ nghiêm trọng: <br>
Ghi nhận 29,255 bản ghi (chiếm 38.92% tổng dữ liệu) xảy ra hiện tượng số lượng hàng hư hỏng vượt quá số lượng hàng nhập kho trong ngày (`damaged_stock > stock_received`). 
Chi tiết ma trận đối chiếu chỉ ra cấu trúc mâu thuẫn như sau:
- Có 28,411 ngày ghi nhận số lượng nhập kho bằng 0 nhưng số lượng hàng hư hỏng lại bằng 2.<br>
- Có 844 ngày ghi nhận số lượng nhập kho bằng 1 nhưng số lượng hàng hư hỏng lại bằng 2.
Về mặt nghiệp vụ quản trị kho, số lượng hư hỏng phát sinh trong ngày không thể vượt ngưỡng lượng hàng nhập nếu tính trên cùng một lô nhật ký, trừ khi biến số này phản ánh lượng hàng tồn kho cũ bị phát hiện lỗi hoặc đây là lỗi gán số liệu ngẫu nhiên không ràng buộc điều kiện từ hệ thống tạo lập dữ liệu.

In [ ]:
# Xuất bảng thống kê mô tả cho các trường định lượng thực sự của kho hàng
df_inventory[['stock_received', 'damaged_stock']].describe().round(2)

,stock_received,damaged_stock
count,75172.00,75172.00
mean,1.96,1.07
std,1.64,1.00
min,0.00,0.00
25%,0.00,0.00
50%,3.00,2.00
75%,3.00,2.00
max,4.00,2.00


Nhận xét về các chỉ số thống kê:
1. Biến `stock_received`: <br>
Số lượng nhập kho trung bình đạt 1.96 đơn vị trên một bản ghi nhật ký. D dải giá trị nhận các mức số nguyên rời rạc từ 0.00 đến 4.00. Khoảng tứ phân vị thể hiện 25% số ngày không phát sinh lượt nhập kho (mức 25% bằng 0.00), và 50% số ngày ghi nhận lượng nhập đạt từ 3.00 đến tối đa 4.00 đơn vị sản phẩm.

2. Biến `damaged_stock`: <br>
Số lượng hàng hóa hư hỏng trung bình đạt 1.07 đơn vị trên một bản ghi nhật ký, độ lệch chuẩn đạt 1.00. Biến số này chỉ nhận hai giá trị rời rạc là 0.00 và 2.00, cho thấy hệ thống phân loại kho hàng đang ghi nhận theo dạng nhị phân cố định thay vì số lượng đếm thực tế biến thiên liên tục.